# Day 05 Tutorial — File Formats, Partitioning & Delta Basics

**Goal:** Parquet partitions and Delta MERGE concepts.

> Delta needs delta-spark locally; built-in on Databricks/Fabric.


### Environment setup
Skip pip install on Databricks/Fabric. Locally you may need: `pip install pyspark pandas`.


In [ ]:
# %pip install pyspark==3.5.1 pandas -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark


In [ ]:
from pathlib import Path
out = Path('data/day05')
out.mkdir(parents=True, exist_ok=True)
df = spark.createDataFrame(
    [
        (1, 'IN', '2024-01-01', 10.0),
        (2, 'IN', '2024-01-01', 20.0),
        (3, 'US', '2024-01-02', 15.0),
        (4, 'US', '2024-01-02', 25.0),
    ],
    ['id', 'country', 'dt', 'amount'],
)


## Partitioned Parquet


In [ ]:
path = str(out / 'sales_parquet')
df.write.mode('overwrite').partitionBy('country', 'dt').parquet(path)
spark.read.parquet(path).filter(F.col('country') == 'IN').show()


## Delta MERGE pattern (optional)


In [ ]:
try:
    target = str(out / 'customers_delta')
    customers = spark.createDataFrame(
        [(1, 'Alice', 'Pune'), (2, 'Bob', 'Delhi')], ['id', 'name', 'city']
    )
    customers.write.format('delta').mode('overwrite').save(target)
    updates = spark.createDataFrame(
        [(2, 'Bob', 'Mumbai'), (3, 'Carol', 'London')], ['id', 'name', 'city']
    )
    from delta.tables import DeltaTable
    dt = DeltaTable.forPath(spark, target)
    (dt.alias('t').merge(updates.alias('s'), 't.id = s.id')
      .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    spark.read.format('delta').load(target).show()
except Exception as e:
    print('Delta not configured:', type(e).__name__, str(e)[:180])
    print('Still learn MERGE for interviews.')


## Interview
- Parquet over CSV for analytics.
- Delta: ACID, time travel, MERGE.
- Partition on filter-friendly columns (often date).
